In [6]:
# Cell 1: imports, helpers, config, model, stage-1 & stage-2

import math, time
from contextlib import ExitStack
from dataclasses import dataclass

import torch, gpytorch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# -------- helpers --------
def now(): return time.strftime("%H:%M:%S")
def log(msg): print(f"[{now()}] {msg}", flush=True)
def has_setting(name: str) -> bool: return hasattr(gpytorch.settings, name)

def ciq_context():
    stack = ExitStack()
    for name, val in [
        ("num_contour_quadrature", 15),
        ("ciq_samples", 16),
        ("ciq_max_iter", 30),
        ("ciq_forward_precision", 0.01),
        ("eval_cg_tolerance", 1e-4),
        ("max_root_decomposition_size", 64),
    ]:
        if has_setting(name):
            stack.enter_context(getattr(gpytorch.settings, name)(val))
    stack.enter_context(gpytorch.settings.fast_pred_var(True))
    return stack

def clone_state_dict(sd):
    out = {}
    for k, v in sd.items():
        out[k] = v.detach().clone() if torch.is_tensor(v) else v
    return out

# -------- toy truth (latent) --------
def f0(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * math.pi * x) + 0.5 * torch.cos(3 * x)

def make_stratified_points_1d(n, low, high, generator, device, dtype):
    i = torch.arange(n, device=device, dtype=dtype)
    u = (i + torch.rand(n, generator=generator, device=device, dtype=dtype)) / n
    x = low + (high - low) * u
    # NOTE: randperm 不传 generator（跟你最新脚本一样）
    perm = torch.randperm(n, device=device)
    return x[perm].unsqueeze(-1)

# -------- config --------
@dataclass
class CFG:
    # data
    n_train: int = 500
    n_test: int = 50
    train_low: float = -2.5
    train_high: float = 2.5
    test_low: float = -2.5
    test_high: float = 2.5

    # model/train
    use_fp64: bool = True
    use_ciq: bool = False
    inducing_points: int = 50
    stage1_epochs: int = 5000
    stage2_epochs: int = 1500   # 对齐 coverage 脚本
    batch_size: int = 500
    lr_adam_stage1: float = 2e-2
    lr_adam_stage2: float = 2e-2
    jitter: float = 1e-2
    seed: int = 13

    # early stopping (Stage-1 only)
    early_stop_patience: int = 60
    early_stop_min_delta: float = 1e-4

cfg = CFG()

# -------- model --------
class SVGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points_std: torch.Tensor):
        M = inducing_points_std.size(0)
        q_dist = gpytorch.variational.CholeskyVariationalDistribution(M)
        q_strat = gpytorch.variational.VariationalStrategy(
            self, inducing_points_std, q_dist, learn_inducing_locations=False
        )
        super().__init__(q_strat)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x_std):
        mean_x = self.mean_module(x_std)
        covar_x = self.covar_module(x_std)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# ---- two-stage param helpers ----
def freeze_all_params(model, likelihood):
    for p in model.parameters():
        p.requires_grad_(False)
    for p in likelihood.parameters():
        p.requires_grad_(False)

def pick_covariance_params_svgp(model):
    """Select ONLY variational covariance parameters (exclude any '*mean*')."""
    trainables = []
    vs = getattr(model, "variational_strategy", None)
    vdist_mod = None
    if vs is not None:
        vdist_mod = getattr(vs, "_variational_distribution", None)
        if vdist_mod is None:
            cand = getattr(vs, "variational_distribution", None)
            if isinstance(cand, torch.nn.Module):
                vdist_mod = cand
    if isinstance(vdist_mod, torch.nn.Module):
        for n, p in vdist_mod.named_parameters(recurse=True):
            low = n.lower()
            if "mean" in low:
                p.requires_grad_(False)
            elif any(k in low for k in ["chol", "std", "var", "covar", "factor"]):
                p.requires_grad_(True)
                trainables.append(p)
            else:
                p.requires_grad_(False)
    if not trainables:
        for n, p in model.named_parameters():
            low = n.lower()
            if (
                "variational_strategy" in low
                and any(
                    k in low
                    for k in [
                        "chol_variational_covar",
                        "raw_covar",
                        "chol_factor",
                        "covar_factor",
                        "variational_stddev",
                        "raw_var",
                        "qchol",
                        "q_scale_tril",
                    ]
                )
                and "mean" not in low
            ):
                p.requires_grad_(True)
                trainables.append(p)
    return trainables

# ---- Stage-1 ----
def stage1_train(
    model,
    likelihood,
    x_train_std,
    y_train,
    lr,
    epochs,
    jitter,
    batch_size,
    early_stop_patience,
    early_stop_min_delta,
):
    """
    Stage-1: α=1, train kernel + variational mean (+likelihood) with ELBO(beta=1).
    """
    N = x_train_std.size(0)
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=N, beta=1.0)

    likelihood.noise_covar.raw_noise.requires_grad_(False)
    opt = torch.optim.Adam(
        [
            {"params": model.variational_parameters()},
            {"params": model.hyperparameters()},
            {"params": likelihood.parameters()},
        ],
        lr=lr,
    )

    loader = DataLoader(
        TensorDataset(x_train_std, y_train),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    model.train()
    likelihood.train()
    best_loss, best_epoch = float("inf"), 0
    best_model_sd = clone_state_dict(model.state_dict())
    best_lik_sd = clone_state_dict(likelihood.state_dict())
    patience = 0

    with gpytorch.settings.cholesky_jitter(jitter):
        for ep in range(1, epochs + 1):
            total, n_batches = 0.0, 0
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                out = model(xb)
                loss = -mll(out, yb)
                if loss.numel() > 1:
                    loss = loss.mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                total += float(loss.item())
                n_batches += 1
            avg_loss = total / max(1, n_batches)
            if best_loss - avg_loss > early_stop_min_delta:
                best_loss, best_epoch = avg_loss, ep
                best_model_sd = clone_state_dict(model.state_dict())
                best_lik_sd = clone_state_dict(likelihood.state_dict())
                patience = 0
            else:
                patience += 1
            if patience >= early_stop_patience:
                break

    model.load_state_dict(best_model_sd)
    likelihood.load_state_dict(best_lik_sd)
    model.eval()
    likelihood.eval()
    return best_epoch, best_loss

# ---- Stage-2 (covariance only) ----
def stage2_train_cov_only(
    model,
    likelihood,
    x_train_std,
    y_train,
    beta,
    lr,
    epochs,
    jitter,
    batch_size,
):
    """
    Stage-2: freeze mean/kernel/likelihood, train ONLY variational covariance with beta=1/alpha.
    """
    N = x_train_std.size(0)
    freeze_all_params(model, likelihood)
    cov_params = pick_covariance_params_svgp(model)
    assert cov_params, "No covariance params found to train in Stage-2."
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=N, beta=beta)
    opt = torch.optim.Adam(cov_params, lr=lr)

    loader = DataLoader(
        TensorDataset(x_train_std, y_train),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    model.train()
    likelihood.train()
    best_loss, best_epoch = float("inf"), 0
    best_model_sd = clone_state_dict(model.state_dict())
    best_lik_sd = clone_state_dict(likelihood.state_dict())

    with gpytorch.settings.cholesky_jitter(jitter):
        for ep in range(1, epochs + 1):
            total, n_batches = 0.0, 0
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                out = model(xb)
                loss = -mll(out, yb)
                if loss.numel() > 1:
                    loss = loss.mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(cov_params, 5.0)
                opt.step()
                total += float(loss.item())
                n_batches += 1
            avg_loss = total / max(1, n_batches)
            if avg_loss < best_loss:
                best_loss, best_epoch = avg_loss, ep
                best_model_sd = clone_state_dict(model.state_dict())
                best_lik_sd = clone_state_dict(likelihood.state_dict())

    model.load_state_dict(best_model_sd)
    likelihood.load_state_dict(best_lik_sd)
    model.eval()
    likelihood.eval()
    return best_epoch, best_loss


In [7]:
# Cell 2: build global data (train/test, symmetric norm, Z)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64 if cfg.use_fp64 else torch.float32)

torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

log(f"Device: {device} | dtype: {torch.get_default_dtype()}")

# global train (raw)
gen = torch.Generator(device=device).manual_seed(cfg.seed)
x_train_raw = make_stratified_points_1d(
    n=cfg.n_train,
    low=cfg.train_low,
    high=cfg.train_high,
    generator=gen,
    device=device,
    dtype=torch.get_default_dtype(),
)
y_train_full = f0(x_train_raw).squeeze(-1)

# global test (raw)
x_test_raw = torch.linspace(
    cfg.test_low,
    cfg.test_high,
    cfg.n_test,
    device=device,
    dtype=torch.get_default_dtype(),
).unsqueeze(1)
y_test_true = f0(x_test_raw).squeeze(-1)

# symmetric normalization
mid = torch.tensor(
    [(cfg.train_low + cfg.train_high) / 2.0],
    device=device,
    dtype=torch.get_default_dtype(),
)
half = torch.tensor(
    [(cfg.train_high - cfg.train_low) / 2.0],
    device=device,
    dtype=torch.get_default_dtype(),
).clamp_min(1e-12)

def to_std(x_raw):
    return (x_raw - mid) / half

x_train_std_full = to_std(x_train_raw)
x_test_std = to_std(x_test_raw)

# fixed inducing points
Z_raw = torch.linspace(
    cfg.train_low,
    cfg.train_high,
    cfg.inducing_points,
    device=device,
    dtype=torch.get_default_dtype(),
).unsqueeze(1)
Z_std = to_std(Z_raw)

log(
    f"Symmetric norm: raw [{cfg.train_low:.2f},{cfg.train_high:.2f}] "
    f"-> std [{float(to_std(torch.tensor([[cfg.train_low]], device=device))):.2f}, "
    f"{float(to_std(torch.tensor([[cfg.train_high]], device=device))):.2f}] (≈ -1,+1)"
)
log(f"Z_std head/tail: {float(Z_std[0]):.3f}, {float(Z_std[-1]):.3f}")


[22:23:40] Device: cuda | dtype: torch.float64
[22:23:40] Symmetric norm: raw [-2.50,2.50] -> std [-1.00, 1.00] (≈ -1,+1)
[22:23:40] Z_std head/tail: -1.000, 1.000


In [8]:
# Cell 3: baseline two-stage (full 500-point training set)

def run_baseline_two_stage(alpha: float, verbose: bool = True):
    """
    Baseline: 全 500 个训练点; Stage-1 (α=1) + Stage-2(beta=1/alpha) on whole data.
    Coverage target: 真 f0(x_test).
    """
    beta = 1.0 / alpha

    model = SVGPModel(Z_std.clone().contiguous()).to(device=device)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
    with torch.no_grad():
        model.covar_module.base_kernel.lengthscale.fill_(1.0)
        model.covar_module.outputscale.fill_(1.0)
        likelihood.noise = torch.as_tensor(3e-3, dtype=torch.get_default_dtype(), device=device)
    likelihood.noise_covar.raw_noise.requires_grad_(False)

    # Stage-1
    with (ciq_context() if cfg.use_ciq else ExitStack()):
        be1, bl1 = stage1_train(
            model,
            likelihood,
            x_train_std=x_train_std_full,
            y_train=y_train_full,
            lr=cfg.lr_adam_stage1,
            epochs=cfg.stage1_epochs,
            jitter=cfg.jitter,
            batch_size=min(cfg.batch_size, x_train_std_full.size(0)),
            early_stop_patience=cfg.early_stop_patience,
            early_stop_min_delta=cfg.early_stop_min_delta,
        )

    # Stage-2
    with (ciq_context() if cfg.use_ciq else ExitStack()):
        be2, bl2 = stage2_train_cov_only(
            model,
            likelihood,
            x_train_std=x_train_std_full,
            y_train=y_train_full,
            beta=beta,
            lr=cfg.lr_adam_stage2,
            epochs=cfg.stage2_epochs,
            jitter=cfg.jitter,
            batch_size=min(cfg.batch_size, x_train_std_full.size(0)),
        )

    with torch.no_grad():
        pf = model(x_test_std)
        mu = pf.mean
        std = pf.variance.sqrt().clamp_min(1e-12)
        lower, upper = mu - 1.96 * std, mu + 1.96 * std
        cover_true = ((y_test_true >= lower) & (y_test_true <= upper)).to(torch.float64)
        mean_cov = float(cover_true.mean().item())
        curve = cover_true.cpu().numpy()

    if verbose:
        log(
            f"[BASELINE α={alpha:.3f}] mean_coverage_true={mean_cov:.4f} | "
            f"Stage-1 best -ELBO={bl1:.4f} @epoch{be1} | "
            f"Stage-2 best -ELBO={bl2:.4f} @epoch{be2}"
        )

    return {
        "mu": mu,
        "std": std,
        "coverage_curve_true": curve,
        "mean_coverage_true": mean_cov,
    }


In [9]:
# Cell 4: single-rep debug calibration (two-split + baseline comparison)

def calibrate_alpha_once_debug(
    alpha: float = 1.0,
    B: int = 100,
    rep_seed: int = 2025,
    verbose: bool = True,
):
    """
    执行一次校准试验（一个 alpha，一个 replication），并同时计算：
      - baseline (full 500 pt, two-stage) 对真 f0 的 coverage
      - Split-2 bootstrap band 对：
          (1) 真 f0
          (2) Split-1 Stage-1 均值 μ_s1
          (3) Split-1 Stage-2 均值 μ_s2
          (4) baseline 均值 μ_base
    """
    beta = 1.0 / alpha

    # ----- 0) baseline on full data -----
    baseline = run_baseline_two_stage(alpha=alpha, verbose=verbose)
    mu_base = baseline["mu"]

    # ----- 1) random split of the 500 train points -----
    gen = torch.Generator(device=device).manual_seed(rep_seed)
    perm = torch.randperm(cfg.n_train, generator=gen, device=device)
    idx1 = perm[: cfg.n_train // 2]
    idx2 = perm[cfg.n_train // 2 : cfg.n_train]

    x1_std, y1 = x_train_std_full[idx1], y_train_full[idx1]
    x2_std, y2 = x_train_std_full[idx2], y_train_full[idx2]

    # ----- 2) Split-1: Stage-1 (pseudo-truth in your current script) -----
    model_s1 = SVGPModel(Z_std.clone().contiguous()).to(device=device)
    lik_s1 = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
    with torch.no_grad():
        model_s1.covar_module.base_kernel.lengthscale.fill_(1.0)
        model_s1.covar_module.outputscale.fill_(1.0)
        lik_s1.noise = torch.as_tensor(3e-3, dtype=torch.get_default_dtype(), device=device)
    lik_s1.noise_covar.raw_noise.requires_grad_(False)

    with (ciq_context() if cfg.use_ciq else ExitStack()):
        be_s1_1, bl_s1_1 = stage1_train(
            model_s1,
            lik_s1,
            x_train_std=x1_std,
            y_train=y1,
            lr=cfg.lr_adam_stage1,
            epochs=cfg.stage1_epochs,
            jitter=cfg.jitter,
            batch_size=min(cfg.batch_size, x1_std.size(0)),
            early_stop_patience=cfg.early_stop_patience,
            early_stop_min_delta=cfg.early_stop_min_delta,
        )

    with torch.no_grad():
        pf_s1 = model_s1(x_test_std)
        mu_s1 = pf_s1.mean        # 这就是你当前脚本里的 mu_truth

    # ----- 3) Split-1: 在同一半数据上跑 Stage-2（看“真两阶段” pseudo-truth） -----
    model_s1_stage2 = SVGPModel(Z_std.clone().contiguous()).to(device=device)
    lik_s1_stage2 = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
    model_s1_stage2.load_state_dict(model_s1.state_dict())
    lik_s1_stage2.load_state_dict(lik_s1.state_dict())

    with (ciq_context() if cfg.use_ciq else ExitStack()):
        be_s1_2, bl_s1_2 = stage2_train_cov_only(
            model_s1_stage2,
            lik_s1_stage2,
            x_train_std=x1_std,
            y_train=y1,
            beta=beta,
            lr=cfg.lr_adam_stage2,
            epochs=cfg.stage2_epochs,
            jitter=cfg.jitter,
            batch_size=min(cfg.batch_size, x1_std.size(0)),
        )

    with torch.no_grad():
        pf_s1_2 = model_s1_stage2(x_test_std)
        mu_s2 = pf_s1_2.mean      # Split-1 的 two-stage 均值

    if verbose:
        log(
            f"[Split-1 α={alpha:.3f}] Stage-1 best -ELBO={bl_s1_1:.4f}@{be_s1_1}, "
            f"Stage-2 best -ELBO={bl_s1_2:.4f}@{be_s1_2}"
        )

    # ----- 4) Split-2: Stage-1 anchor -----
    model_s2_anchor = SVGPModel(Z_std.clone().contiguous()).to(device=device)
    lik_s2_anchor = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
    with torch.no_grad():
        model_s2_anchor.covar_module.base_kernel.lengthscale.fill_(1.0)
        model_s2_anchor.covar_module.outputscale.fill_(1.0)
        lik_s2_anchor.noise = torch.as_tensor(3e-3, dtype=torch.get_default_dtype(), device=device)
    lik_s2_anchor.noise_covar.raw_noise.requires_grad_(False)

    with (ciq_context() if cfg.use_ciq else ExitStack()):
        be_s2_1, bl_s2_1 = stage1_train(
            model_s2_anchor,
            lik_s2_anchor,
            x_train_std=x2_std,
            y_train=y2,
            lr=cfg.lr_adam_stage1,
            epochs=cfg.stage1_epochs,
            jitter=cfg.jitter,
            batch_size=min(cfg.batch_size, x2_std.size(0)),
            early_stop_patience=cfg.early_stop_patience,
            early_stop_min_delta=cfg.early_stop_min_delta,
        )

    s2_model_sd = clone_state_dict(model_s2_anchor.state_dict())
    s2_lik_sd = clone_state_dict(lik_s2_anchor.state_dict())

    if verbose:
        log(
            f"[Split-2 α={alpha:.3f}] Stage-1 anchor best -ELBO={bl_s2_1:.4f}@{be_s2_1}"
        )

    # ----- 5) Bootstrap on Split-2 with Stage-2(cov-only) -----
    J = x_test_std.size(0)
    indicators_true = np.zeros((B, J), dtype=np.float64)
    indicators_mu_s1 = np.zeros((B, J), dtype=np.float64)
    indicators_mu_s2 = np.zeros((B, J), dtype=np.float64)
    indicators_mu_base = np.zeros((B, J), dtype=np.float64)
    avg_bandwidth = np.zeros(B, dtype=np.float64)

    idx_all = torch.arange(x2_std.size(0), device=device)

    for b in range(B):
        if verbose and (b % 20 == 0 or b == B - 1):
            log(f"[Calib α={alpha:.3f}] rep_seed={rep_seed} bootstrap {b+1}/{B}")

        boot_idx = idx_all[
            torch.randint(
                low=0,
                high=x2_std.size(0),
                size=(x2_std.size(0),),
                generator=gen,
                device=device,
            )
        ]
        xb, yb = x2_std[boot_idx], y2[boot_idx]

        model_b = SVGPModel(Z_std.clone().contiguous()).to(device=device)
        lik_b = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
        model_b.load_state_dict(s2_model_sd)
        lik_b.load_state_dict(s2_lik_sd)

        with (ciq_context() if cfg.use_ciq else ExitStack()):
            be_b, bl_b = stage2_train_cov_only(
                model_b,
                lik_b,
                x_train_std=xb,
                y_train=yb,
                beta=beta,
                lr=cfg.lr_adam_stage2,
                epochs=cfg.stage2_epochs,
                jitter=cfg.jitter,
                batch_size=min(cfg.batch_size, xb.size(0)),
            )

        with torch.no_grad():
            pf_b = model_b(x_test_std)
            mu_b = pf_b.mean
            std_b = pf_b.variance.sqrt().clamp_min(1e-12)
            lower, upper = mu_b - 1.96 * std_b, mu_b + 1.96 * std_b

            cover_true = ((y_test_true >= lower) & (y_test_true <= upper)).to(torch.float64)
            cover_s1   = ((mu_s1        >= lower) & (mu_s1        <= upper)).to(torch.float64)
            cover_s2   = ((mu_s2        >= lower) & (mu_s2        <= upper)).to(torch.float64)
            cover_base = ((mu_base      >= lower) & (mu_base      <= upper)).to(torch.float64)

            indicators_true[b, :] = cover_true.cpu().numpy()
            indicators_mu_s1[b, :] = cover_s1.cpu().numpy()
            indicators_mu_s2[b, :] = cover_s2.cpu().numpy()
            indicators_mu_base[b, :] = cover_base.cpu().numpy()
            avg_bandwidth[b] = float((1.96 * std_b).mean().item())

    # ----- 6) aggregate -----
    def agg(ind_mat):
        C_point = ind_mat.mean(axis=0)
        return float(C_point.mean()), float(C_point.min()), C_point

    mean_cov_true, min_cov_true, curve_true = agg(indicators_true)
    mean_cov_s1,   min_cov_s1,   curve_s1   = agg(indicators_mu_s1)
    mean_cov_s2,   min_cov_s2,   curve_s2   = agg(indicators_mu_s2)
    mean_cov_base2,min_cov_base2,curve_base2= agg(indicators_mu_base)

    bw_mean = float(avg_bandwidth.mean())

    res = {
        "alpha": alpha,
        "B": B,
        "rep_seed": rep_seed,
        "x": x_test_raw.squeeze(-1).cpu().numpy(),

        # baseline two-stage on full data
        "baseline_mean_coverage_true": baseline["mean_coverage_true"],
        "baseline_curve_true": baseline["coverage_curve_true"],

        # calibration (Split-2 bands) vs different targets
        "mean_cov_split2_true": mean_cov_true,
        "min_cov_split2_true": min_cov_true,
        "curve_split2_true": curve_true,

        "mean_cov_split2_mu_s1": mean_cov_s1,
        "min_cov_split2_mu_s1": min_cov_s1,
        "curve_split2_mu_s1": curve_s1,

        "mean_cov_split2_mu_s2": mean_cov_s2,
        "min_cov_split2_mu_s2": min_cov_s2,
        "curve_split2_mu_s2": curve_s2,

        "mean_cov_split2_mu_base": mean_cov_base2,
        "min_cov_split2_mu_base": min_cov_base2,
        "curve_split2_mu_base": curve_base2,

        "mean_bandwidth": bw_mean,
    }
    return res


In [11]:
# Cell 5: run one debug calibration experiment for alpha=1.0

alpha_debug = 1.0
B_debug = 100   # 可以先用 50/20 快速看一下
seed_debug = 2025

res = calibrate_alpha_once_debug(
    alpha=alpha_debug,
    B=B_debug,
    rep_seed=seed_debug,
    verbose=True,
)

# --- 表格：把 baseline vs calibration 放在一起看 ---
row = {
    "alpha": alpha_debug,
    "B": B_debug,
    "baseline_mean_cov_true": res["baseline_mean_coverage_true"],
    "split2_cov_true":        res["mean_cov_split2_true"],
    "split2_cov_mu_s1":       res["mean_cov_split2_mu_s1"],
    "split2_cov_mu_s2":       res["mean_cov_split2_mu_s2"],
    "split2_cov_mu_base":     res["mean_cov_split2_mu_base"],
    "mean_bandwidth":         res["mean_bandwidth"],
}
df_debug = pd.DataFrame([row])
display(df_debug)


[22:24:40] [BASELINE α=1.000] mean_coverage_true=0.8000 | Stage-1 best -ELBO=-1.6690 @epoch633 | Stage-2 best -ELBO=-1.6765 @epoch1362
[22:25:23] [Split-1 α=1.000] Stage-1 best -ELBO=-1.5622@1191, Stage-2 best -ELBO=-1.5683@1499
[22:25:37] [Split-2 α=1.000] Stage-1 anchor best -ELBO=-1.4774@776
[22:25:37] [Calib α=1.000] rep_seed=2025 bootstrap 1/100
[22:32:40] [Calib α=1.000] rep_seed=2025 bootstrap 21/100


KeyboardInterrupt: 

In [ ]:
# Cell 6: plot point-wise coverage curves under different notions of "truth"

xs = res["x"]

plt.figure(figsize=(8, 4.5))
plt.plot(xs, res["baseline_curve_true"], label=f"Baseline vs f0 (mean={res['baseline_mean_coverage_true']:.3f})")
plt.plot(xs, res["curve_split2_true"],  label=f"Split-2 band vs f0 (mean={res['mean_cov_split2_true']:.3f})")
plt.hlines(0.95, xs.min(), xs.max(), linestyles="--", colors="k", label="0.95")
plt.xlabel("x")
plt.ylabel("coverage")
plt.title(f"Point-wise coverage (alpha={alpha_debug:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(xs, res["curve_split2_mu_s1"], label=f"Split-2 band vs Split-1 μ(Stage1) (mean={res['mean_cov_split2_mu_s1']:.3f})")
plt.plot(xs, res["curve_split2_mu_s2"], label=f"Split-2 band vs Split-1 μ(Stage1+Stage2) (mean={res['mean_cov_split2_mu_s2']:.3f})")
plt.plot(xs, res["curve_split2_mu_base"], label=f"Split-2 band vs baseline μ(full two-stage) (mean={res['mean_cov_split2_mu_base']:.3f})")
plt.hlines(0.95, xs.min(), xs.max(), linestyles="--", colors="k", label="0.95")
plt.xlabel("x")
plt.ylabel("coverage")
plt.title(f"Point-wise self-coverage under different 'truth' choices (alpha={alpha_debug:.3f})")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 7 (optional): multiple replications for one alpha in notebook

def run_replications_debug_one_alpha(
    alpha: float,
    B: int,
    R: int,
    base_seed: int = 2025,
):
    rows = []
    for r in range(R):
        seed_r = base_seed + r
        log(f"[DEBUG RUN] alpha={alpha:.3f}, rep={r+1}/{R}, seed={seed_r}")
        res = calibrate_alpha_once_debug(
            alpha=alpha,
            B=B,
            rep_seed=seed_r,
            verbose=False,
        )
        row = {
            "alpha": alpha,
            "B": B,
            "rep": r,
            "seed": seed_r,
            "baseline_mean_cov_true": res["baseline_mean_coverage_true"],
            "split2_cov_true":        res["mean_cov_split2_true"],
            "split2_cov_mu_s1":       res["mean_cov_split2_mu_s1"],
            "split2_cov_mu_s2":       res["mean_cov_split2_mu_s2"],
            "split2_cov_mu_base":     res["mean_cov_split2_mu_base"],
            "mean_bandwidth":         res["mean_bandwidth"],
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    return df

# Example: small R for alpha=1.0
df_reps = run_replications_debug_one_alpha(alpha=1.0, B=50, R=5)
display(df_reps)
print("\nAggregated means:")
display(df_reps.mean(numeric_only=True))
